<a href="https://colab.research.google.com/github/alimovscott/cloneGPT/blob/main/clone_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install transformers torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.8 MB/s eta 0:00:00


In [17]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

In [21]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print("Vocab size:", tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)


# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    # dtype=torch.bfloat16

    )
#

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [23]:
# Before Fine-tuning
promt = "Explain what a tokenizer is? "
# promt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(
    promt,
    return_tensors='pt'
).to(model.device)

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

  print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is? 
- 5.0.0: The tokenizer generates a list of tokens by splitting the text into individual words, based on the rules of the tokenizer. These tokens are then used to generate a series of character indices, which can be passed to the character tokenizer to generate character indices of the actual characters in the text.
- 5.0.1: The tokenizer


In [24]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() )

total_params = count_parameters(model)
print(f'Total parameters: {total_params:,}')



Total parameters: 615,606,272
